# Second-Hand Car Price Prediction & Classification

**Course:** Machine Learning  
**Dataset:** Used Cars Dataset (4,008 rows, 9 features, 53 brands, years 1992–2024)  
**Objective:** Predict the selling price (regression) and classify it into Low / Medium / High (classification) using scikit-learn pipelines.

---

## Table of Contents
1. [Unit I – Data Preparation](#unit1)
2. [Unit II – EDA & Feature Engineering](#unit2)
3. [Unit III – Regression Models](#unit3)
4. [Unit IV – Classification Models](#unit4)

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    BrandExtractor, CarAgeFeatureEngineer, iqr_bounds,
    COL_CAR_NAME, COL_YEAR, COL_SELLING, COL_PRESENT,
    COL_KMS, COL_FUEL, COL_SELLER, COL_TRANS, COL_OWNER,
    COL_BRAND, COL_CAR_AGE, COL_KM_PER_YR, CURRENT_YEAR
)

RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')
print('sklearn version:', sklearn.__version__)
print('pandas  version:', pd.__version__)
print('numpy   version:', np.__version__)

<a id='unit1'></a>
## UNIT I – Data Preparation

In this unit we:
- Load the dataset and inspect shape, dtypes, and missing values
- Coerce numeric columns and impute missing values
- Engineer `Brand`, `Car_Age`, and `Km_Per_Year` features
- Scale numeric features (preview only — the pipeline handles this during training)
- Split into 80 / 20 train-test sets

In [ ]:
# 1.1  Load Data
DATA_PATH = PROJECT_ROOT / 'data' / 'car_data_new.csv'
df_raw = pd.read_csv(DATA_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# 1.2  Inspect dtypes and missing values
print('--- dtypes ---')
print(df_raw.dtypes)
print('\n--- missing values ---')
print(df_raw.isnull().sum())
df_raw.describe()

In [ ]:
# 1.3  Clean numeric columns and impute missing values
df = df_raw.copy()
numeric_cols = [COL_YEAR, COL_SELLING, COL_PRESENT, COL_KMS, COL_OWNER]
for c in numeric_cols:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.extract(r'([\d.]+)', expand=False),
        errors='coerce'
    )

for c in [COL_YEAR, COL_PRESENT, COL_KMS, COL_OWNER]:
    df[c] = df[c].fillna(df[c].median())
for c in [COL_FUEL, COL_SELLER, COL_TRANS]:
    df[c] = df[c].fillna(df[c].mode()[0])
df = df.dropna(subset=[COL_SELLING])

print('Missing after cleaning:')
print(df.isnull().sum())
print(f'\nShape after cleaning: {df.shape}')

In [ ]:
# 1.4  Feature Engineering (for EDA preview)
from sklearn.preprocessing import LabelEncoder

df_enc = df.copy()
le = LabelEncoder()
for c in [COL_FUEL, COL_SELLER, COL_TRANS]:
    df_enc[c + '_enc'] = le.fit_transform(df_enc[c].astype(str))

df_enc[COL_BRAND]     = df_enc[COL_CAR_NAME].str.strip().str.split().str[0].str.lower().str.capitalize()
df_enc[COL_CAR_AGE]   = (CURRENT_YEAR - df_enc[COL_YEAR]).clip(lower=0)
df_enc[COL_KM_PER_YR] = df_enc[COL_KMS] / df_enc[COL_CAR_AGE].replace(0, 1)

print('Feature engineered columns:')
df_enc[[COL_BRAND, COL_CAR_AGE, COL_KM_PER_YR]].head(3)

In [ ]:
# 1.5  Scale numeric features (preview — pipeline handles this during training)
from sklearn.preprocessing import StandardScaler

num_cols = [COL_YEAR, COL_PRESENT, COL_KMS, COL_OWNER, COL_CAR_AGE, COL_KM_PER_YR]
scaler = StandardScaler()
scaled = scaler.fit_transform(df_enc[num_cols])
print('Scaled feature stats (mean~0, std~1):')
pd.DataFrame(scaled, columns=num_cols).describe().round(3)

In [ ]:
# 1.6  Train-Test Split (preview — final split is done after IQR cleaning in Unit III)
from sklearn.model_selection import train_test_split

FEATURE_COLS = [COL_CAR_NAME, COL_YEAR, COL_PRESENT, COL_KMS, COL_FUEL, COL_SELLER, COL_TRANS, COL_OWNER]
X_raw = df[FEATURE_COLS].copy()
y     = df[COL_SELLING].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train_raw.shape}, Test: {X_test_raw.shape}')

<a id='unit2'></a>
## UNIT II – EDA & Feature Engineering

We explore:
- Price distribution (raw and log-transformed)
- Price vs Year and Kms Driven
- Average price by Brand, Fuel type, and Transmission
- Correlation heatmap
- IQR-based outlier detection and removal
- Feature importance using Random Forest (exploratory)
- Class imbalance check for Low / Medium / High price categories

In [ ]:
# 2.1  Price Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_enc[COL_SELLING], bins=30, color='#4C72B0', edgecolor='white')
axes[0].set_title('Selling Price Distribution')
axes[0].set_xlabel('Price (Lakhs)')
axes[1].hist(np.log1p(df_enc[COL_SELLING]), bins=30, color='#DD8452', edgecolor='white')
axes[1].set_title('Log(Selling Price) Distribution')
axes[1].set_xlabel('log(Price + 1)')
plt.tight_layout()
plt.show()

In [ ]:
# 2.2  Price vs Year and Kms Driven
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df_enc[COL_YEAR], df_enc[COL_SELLING], alpha=0.4, color='#4C72B0')
axes[0].set_title('Price vs Year')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Selling Price (Lakhs)')
axes[1].scatter(df_enc[COL_KMS], df_enc[COL_SELLING], alpha=0.4, color='#DD8452')
axes[1].set_title('Price vs Kms Driven')
axes[1].set_xlabel('Kms Driven'); axes[1].set_ylabel('Selling Price (Lakhs)')
plt.tight_layout()
plt.show()

In [ ]:
# 2.3  Average Selling Price by Brand (top 20)
brand_price = df_enc.groupby(COL_BRAND)[COL_SELLING].mean().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(14, 5))
brand_price.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Average Selling Price by Brand (Top 20)')
ax.set_ylabel('Avg Price (Lakhs)')
ax.set_xlabel('Brand')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 2.4  Average Price by Fuel Type and Transmission
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df_enc.groupby(COL_FUEL)[COL_SELLING].mean().sort_values().plot(
    kind='barh', ax=axes[0], color='#55A868')
axes[0].set_title('Avg Price by Fuel Type')
df_enc.groupby(COL_TRANS)[COL_SELLING].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='#C44E52')
axes[1].set_title('Avg Price by Transmission')
plt.tight_layout()
plt.show()

In [ ]:
# 2.5  Correlation Heatmap
num_cols_corr = [COL_YEAR, COL_SELLING, COL_PRESENT, COL_KMS, COL_OWNER, COL_CAR_AGE, COL_KM_PER_YR]
corr = df_enc[num_cols_corr].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, square=True)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# 2.6  IQR Outlier Removal
df_clean = df_enc.copy()
before = len(df_clean)
for col in [COL_SELLING, COL_KMS, COL_PRESENT]:
    lo, hi = iqr_bounds(df_clean[col])
    df_clean = df_clean[(df_clean[col] >= lo) & (df_clean[col] <= hi)]
print(f'Rows before IQR filter: {before}')
print(f'Rows after  IQR filter: {len(df_clean)}')
print(f'Removed: {before - len(df_clean)} outliers')

In [ ]:
# 2.7  Feature Importance (exploratory — using Random Forest for ranking only)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

NUM_F = [COL_YEAR, COL_PRESENT, COL_KMS, COL_OWNER, COL_CAR_AGE, COL_KM_PER_YR]
CAT_F = [COL_BRAND, COL_FUEL, COL_SELLER, COL_TRANS]

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), NUM_F),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT_F),
], remainder='drop')

fi_pipe = Pipeline([
    ('brand', BrandExtractor()),
    ('age',   CarAgeFeatureEngineer()),
    ('prep',  prep),
    ('rf',    RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE))
])

X_fi = df_clean[[COL_CAR_NAME, COL_YEAR, COL_PRESENT, COL_KMS, COL_FUEL, COL_SELLER, COL_TRANS, COL_OWNER]]
y_fi = df_clean[COL_SELLING].values
fi_pipe.fit(X_fi, y_fi)

ohe_names = fi_pipe.named_steps['prep'].named_transformers_['cat'].named_steps['ohe'].get_feature_names_out(CAT_F)
feature_names = NUM_F + list(ohe_names)
importances   = fi_pipe.named_steps['rf'].feature_importances_

fi_df = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 5))
fi_df.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='white')
ax.set_title('Top 15 Feature Importances (Random Forest — EDA only)')
ax.set_ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 2.8  Price Category Distribution (Low / Medium / High)
df_clean = df_clean.copy()
df_clean['price_category'] = pd.qcut(
    df_clean[COL_SELLING], q=3, labels=['Low', 'Medium', 'High']
)
print('Price category distribution:')
print(df_clean['price_category'].value_counts())

fig, ax = plt.subplots(figsize=(6, 4))
df_clean['price_category'].value_counts().plot(kind='bar', ax=ax, color=['#55A868', '#4C72B0', '#C44E52'], edgecolor='white')
ax.set_title('Price Category Distribution')
ax.set_xlabel('Category'); ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

<a id='unit3'></a>
## UNIT III – Regression Models

We build and compare the following regression models to predict `Selling_Price`:

| Model | Notes |
|---|---|
| Simple Linear Regression | Baseline — uses only `Present_Price` |
| Multiple Linear Regression | All features |
| Polynomial Regression (deg=2) | Captures non-linear age/mileage curve — **best model** |
| Ridge Regression | L2 regularisation |
| Lasso Regression | L1 regularisation |
| ElasticNet | Combined L1 + L2 |
| Random Forest Regressor | Ensemble baseline |

Evaluation metrics: MAE, RMSE, R², 5-Fold CV R²

In [ ]:
# 3.0  Shared pipeline building blocks + final train/test split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

NUM_FEATURES = [COL_YEAR, COL_PRESENT, COL_KMS, COL_OWNER, COL_CAR_AGE, COL_KM_PER_YR]
CAT_FEATURES = [COL_BRAND, COL_FUEL, COL_SELLER, COL_TRANS]
FEATURE_COLS = [COL_CAR_NAME, COL_YEAR, COL_PRESENT, COL_KMS, COL_FUEL, COL_SELLER, COL_TRANS, COL_OWNER]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUM_FEATURES),
    ('cat', categorical_transformer, CAT_FEATURES),
], remainder='drop')

def make_pipeline(model):
    return Pipeline([
        ('brand',      BrandExtractor()),
        ('age',        CarAgeFeatureEngineer()),
        ('preprocess', preprocessor),
        ('model',      model),
    ])

X = df_clean[FEATURE_COLS].copy()
y_reg = df_clean[COL_SELLING].values
y_clf = df_clean['price_category'].values

X_train, X_test, yr_train, yr_test = train_test_split(X, y_reg, test_size=0.2, random_state=RANDOM_STATE)
_,       _,      yc_train, yc_test = train_test_split(X, y_clf, test_size=0.2, random_state=RANDOM_STATE)
print(f'Train size: {len(X_train)},  Test size: {len(X_test)}')

In [ ]:
# 3.1  Evaluation helper
reg_results = []

def eval_reg(name, pipe, X_tr, y_tr, X_te, y_te):
    pipe.fit(X_tr, y_tr)
    yp   = pipe.predict(X_te)
    mae  = mean_absolute_error(y_te, yp)
    mse  = mean_squared_error(y_te, yp)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_te, yp)
    cv   = cross_val_score(pipe, X_tr, y_tr, cv=5, scoring='r2').mean()
    print(f'  {name:28s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}  CV-R2={cv:.4f}')
    reg_results.append({'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4),
                        'R2': round(r2,4), 'CV_R2': round(cv,4)})
    return pipe, yp

In [ ]:
# 3.2  Simple Linear Regression (baseline — single feature)
_, _ = eval_reg(
    'Simple Linear Regression',
    make_pipeline(LinearRegression()),
    X_train[[COL_CAR_NAME, COL_YEAR, COL_PRESENT, COL_KMS, COL_FUEL, COL_SELLER, COL_TRANS, COL_OWNER]],
    yr_train,
    X_test[[COL_CAR_NAME, COL_YEAR, COL_PRESENT, COL_KMS, COL_FUEL, COL_SELLER, COL_TRANS, COL_OWNER]],
    yr_test
)

In [ ]:
# 3.3  Multiple Linear Regression
pipe_mlr, _ = eval_reg('Multiple Linear Regression', make_pipeline(LinearRegression()),
                        X_train, yr_train, X_test, yr_test)

In [ ]:
# 3.4  Polynomial Regression (degree = 2)  <-- BEST MODEL
poly_pipe = Pipeline([
    ('brand',      BrandExtractor()),
    ('age',        CarAgeFeatureEngineer()),
    ('preprocess', preprocessor),
    ('poly',       PolynomialFeatures(degree=2, include_bias=False)),
    ('model',      LinearRegression()),
])
pipe_poly, yp_poly = eval_reg('Polynomial Regression (deg=2)', poly_pipe,
                               X_train, yr_train, X_test, yr_test)

In [ ]:
# 3.5  Ridge Regression (alpha search)
alphas = [0.01, 0.1, 1, 10, 100]
best_a, best_s = None, -999
for a in alphas:
    s = cross_val_score(make_pipeline(Ridge(alpha=a, random_state=RANDOM_STATE)),
                        X_train, yr_train, cv=5, scoring='r2').mean()
    if s > best_s:
        best_s, best_a = s, a
pipe_ridge, _ = eval_reg(f'Ridge (alpha={best_a})',
                          make_pipeline(Ridge(alpha=best_a, random_state=RANDOM_STATE)),
                          X_train, yr_train, X_test, yr_test)

In [ ]:
# 3.6  Lasso Regression (alpha search)
best_a, best_s = None, -999
for a in alphas:
    s = cross_val_score(make_pipeline(Lasso(alpha=a, random_state=RANDOM_STATE, max_iter=5000)),
                        X_train, yr_train, cv=5, scoring='r2').mean()
    if s > best_s:
        best_s, best_a = s, a
pipe_lasso, _ = eval_reg(f'Lasso (alpha={best_a})',
                          make_pipeline(Lasso(alpha=best_a, random_state=RANDOM_STATE, max_iter=5000)),
                          X_train, yr_train, X_test, yr_test)

In [ ]:
# 3.7  ElasticNet
pipe_en, _ = eval_reg('ElasticNet',
                       make_pipeline(ElasticNet(alpha=0.1, l1_ratio=0.5,
                                                random_state=RANDOM_STATE, max_iter=5000)),
                       X_train, yr_train, X_test, yr_test)

In [ ]:
# 3.8  Random Forest Regressor
pipe_rfr, _ = eval_reg('Random Forest Regressor',
                        make_pipeline(RandomForestRegressor(n_estimators=100, max_depth=10,
                                                            random_state=RANDOM_STATE, n_jobs=-1)),
                        X_train, yr_train, X_test, yr_test)

In [ ]:
# 3.9  Regression Model Comparison Table
reg_df = pd.DataFrame(reg_results)
best_r2 = reg_df['R2'].max()
print('\nRegression Results:')
print(reg_df.to_string(index=False))
print(f'\nBest model: {reg_df.loc[reg_df["R2"].idxmax(), "Model"]}  R2={best_r2:.4f}')

In [ ]:
# 3.10  Actual vs Predicted & Residuals — Polynomial Regression (best model)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(yr_test, yp_poly, alpha=0.6, color='#4C72B0')
lims = [min(yr_test.min(), yp_poly.min()), max(yr_test.max(), yp_poly.max())]
axes[0].plot(lims, lims, 'r--')
axes[0].set_xlabel('Actual Price (Lakhs)')
axes[0].set_ylabel('Predicted Price (Lakhs)')
axes[0].set_title('Actual vs Predicted — Polynomial Regression (deg=2)')

residuals = yr_test - yp_poly
axes[1].scatter(yp_poly, residuals, alpha=0.6, color='#DD8452')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot — Polynomial Regression (deg=2)')

plt.tight_layout()
plt.show()

In [ ]:
# 3.11  Regression Model Comparison — R² Bar Chart
reg_df_sorted = reg_df.sort_values('R2', ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(reg_df_sorted['Model'], reg_df_sorted['R2'], color='#4C72B0', edgecolor='white')
ax.set_xlabel('R² Score')
ax.set_title('Regression Model Comparison (R² Score)')
for bar, val in zip(bars, reg_df_sorted['R2']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 3.12  Save best regressor (Polynomial Regression)
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(pipe_poly, MODELS_DIR / 'regressor.joblib', compress=3)
size = (MODELS_DIR / 'regressor.joblib').stat().st_size / 1024
print(f'Saved regressor.joblib  ({size:.1f} KB)')
print(f'Best regressor: Polynomial Regression (deg=2)  R2={r2_score(yr_test, yp_poly):.4f}')

<a id='unit4'></a>
## UNIT IV – Classification Models

We classify the price into **Low / Medium / High** using the same preprocessing pipeline.

Models compared:

| Model | Notes |
|---|---|
| Logistic Regression | Linear probabilistic baseline |
| K-Nearest Neighbours (k=5) | Distance-based |
| Decision Tree (max_depth=8) | Threshold-based — **best model** |
| Random Forest Classifier | Ensemble |
| SVM (RBF kernel) | Margin-based |
| Gaussian Naive Bayes | Probabilistic |

Evaluation metrics: Accuracy, Macro Precision, Macro Recall, Macro F1, ROC-AUC (OVR)

In [ ]:
# 4.0  Classification imports and evaluation helper
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize

clf_results = []

def eval_clf(name, pipe, X_tr, y_tr, X_te, y_te):
    pipe.fit(X_tr, y_tr)
    yp  = pipe.predict(X_te)
    acc = accuracy_score(y_te, yp)
    rpt = classification_report(y_te, yp, output_dict=True, zero_division=0)
    mp  = rpt['macro avg']['precision']
    mr  = rpt['macro avg']['recall']
    mf1 = rpt['macro avg']['f1-score']
    try:
        yprob = pipe.predict_proba(X_te)
        ybin  = label_binarize(y_te, classes=pipe.classes_)
        auc   = roc_auc_score(ybin, yprob, multi_class='ovr', average='macro')
    except Exception:
        auc = float('nan')
    print(f'  {name:28s}  Acc={acc:.4f}  F1={mf1:.4f}  AUC={auc:.4f}')
    clf_results.append({'Model': name, 'Accuracy': round(acc,4), 'Precision': round(mp,4),
                        'Recall': round(mr,4), 'F1': round(mf1,4),
                        'ROC_AUC': round(auc,4) if not np.isnan(auc) else None})
    return pipe

In [ ]:
# 4.1  Train and evaluate all classifiers
classifiers = [
    ('Logistic Regression',   make_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))),
    ('KNN (k=5)',             make_pipeline(KNeighborsClassifier(n_neighbors=5))),
    ('Decision Tree',         make_pipeline(DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE))),
    ('Random Forest Clf',     make_pipeline(RandomForestClassifier(n_estimators=100, max_depth=10,
                                                                    random_state=RANDOM_STATE, n_jobs=-1))),
    ('SVM (RBF)',             make_pipeline(SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE))),
    ('Gaussian Naive Bayes',  make_pipeline(GaussianNB())),
]

trained_clfs = {}
for name, pipe in classifiers:
    trained_clfs[name] = eval_clf(name, pipe, X_train, yc_train, X_test, yc_test)

In [ ]:
# 4.2  Classification Comparison Table
clf_df = pd.DataFrame(clf_results)
print('\nClassification Results:')
print(clf_df.to_string(index=False))
best_name = clf_df.loc[clf_df['F1'].idxmax(), 'Model']
print(f'\nBest classifier: {best_name}  F1={clf_df["F1"].max():.4f}')

In [ ]:
# 4.3  Detailed Classification Report — Decision Tree (best model)
best_pipe = trained_clfs[best_name]
yp_best   = best_pipe.predict(X_test)
print(f'Classification Report — {best_name}')
print(classification_report(yc_test, yp_best, zero_division=0))

In [ ]:
# 4.4  Confusion Matrix
LABELS = ['Low', 'Medium', 'High']
cm = confusion_matrix(yc_test, yp_best, labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# 4.5  ROC Curves (One-vs-Rest)
color_map   = {'Low': '#4C72B0', 'Medium': '#DD8452', 'High': '#55A868'}
clf_classes = list(best_pipe.classes_)
fig, ax     = plt.subplots(figsize=(8, 6))

for label in LABELS:
    if label in clf_classes:
        idx        = clf_classes.index(label)
        y_prob_col = best_pipe.predict_proba(X_test)[:, idx]
        y_bin      = (np.array(yc_test) == label).astype(int)
        fpr, tpr, _ = roc_curve(y_bin, y_prob_col)
        auc_val    = roc_auc_score(y_bin, y_prob_col)
        ax.plot(fpr, tpr, color=color_map.get(label), label=f'{label} (AUC={auc_val:.2f})')

ax.plot([0,1],[0,1],'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — {best_name}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4.6  Classification Model Comparison — Macro F1 Bar Chart
clf_df_sorted = clf_df.sort_values('F1', ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(clf_df_sorted['Model'], clf_df_sorted['F1'], color='#55A868', edgecolor='white')
ax.set_xlabel('Macro F1 Score')
ax.set_title('Classification Model Comparison (Macro F1 Score)')
for bar, val in zip(bars, clf_df_sorted['F1']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 4.7  Save best classifier + metadata.json
import json

joblib.dump(best_pipe, MODELS_DIR / 'classifier.joblib', compress=3)
clf_size = (MODELS_DIR / 'classifier.joblib').stat().st_size / 1024
print(f'Saved classifier.joblib  ({clf_size:.1f} KB)')
print(f'Best classifier: {best_name}  F1={clf_df["F1"].max():.4f}')

brands_list  = sorted(df_clean[COL_BRAND].dropna().unique().tolist())
fuels_list   = sorted(df_clean[COL_FUEL].dropna().unique().tolist())
sellers_list = sorted(df_clean[COL_SELLER].dropna().unique().tolist())
trans_list   = sorted(df_clean[COL_TRANS].dropna().unique().tolist())
owners_list  = sorted(df_clean[COL_OWNER].dropna().unique().tolist())
q33 = float(df_clean[COL_SELLING].quantile(1/3))
q67 = float(df_clean[COL_SELLING].quantile(2/3))

metadata = {
    'sklearn_version': sklearn.__version__,
    'current_year': CURRENT_YEAR,
    'feature_list': FEATURE_COLS,
    'categorical_options': {
        COL_BRAND:  brands_list,
        COL_FUEL:   fuels_list,
        COL_SELLER: sellers_list,
        COL_TRANS:  trans_list,
        COL_OWNER:  [int(o) for o in owners_list],
    },
    'numeric_ranges': {
        COL_YEAR:    {'min': int(df_clean[COL_YEAR].min()),    'max': int(df_clean[COL_YEAR].max()),    'default': int(df_clean[COL_YEAR].median())},
        COL_PRESENT: {'min': float(df_clean[COL_PRESENT].min()), 'max': float(df_clean[COL_PRESENT].max()), 'default': float(df_clean[COL_PRESENT].median())},
        COL_KMS:     {'min': float(df_clean[COL_KMS].min()),   'max': float(df_clean[COL_KMS].max()),   'default': float(df_clean[COL_KMS].median())},
        COL_OWNER:   {'min': int(df_clean[COL_OWNER].min()),   'max': int(df_clean[COL_OWNER].max()),   'default': int(df_clean[COL_OWNER].median())},
    },
    'category_thresholds': {
        'Low_to_Medium_lakhs':  round(q33, 2),
        'Medium_to_High_lakhs': round(q67, 2),
    },
    'regression_results':     reg_df.to_dict(orient='records'),
    'classification_results': clf_df.to_dict(orient='records'),
    'best_regressor':  'Polynomial Regression (deg=2)',
    'best_classifier': best_name,
}

with open(MODELS_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved metadata.json')